# Homework Week 5

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [3]:
num = 100000 
 
difficulty = np.random.uniform(0, 1, (num,)) 
 
speed = np.maximum(np.random.normal(15, 5, (num, )) - difficulty * 10, 0) 
 
accident = np.minimum(np.maximum(0.03 * speed + 0.4 * difficulty + np.random.normal(0, 0.3, (num,)), 0), 1) 
 
df = pd.DataFrame({'difficulty': difficulty, 'speed': speed, 'accident': accident}) 

In [4]:
coefs = []
for _ in range(100): # 100 datasets
    difficulty = np.random.uniform(0, 1, (num,))
    speed = np.maximum(np.random.normal(15, 5, (num, )) - difficulty * 10, 0)
    # Regress speed (Y) on difficulty (X)
    X = sm.add_constant(difficulty)
    model = sm.OLS(speed, X)
    results = model.fit()
    coefs.append(results.params[1])
np.mean(coefs)

np.float64(-9.651897151233618)

In [6]:
coefs_xz = []
for _ in range(100):
    difficulty = np.random.uniform(0, 1, (num,))
    speed = np.maximum(np.random.normal(15, 5, (num, )) - difficulty * 10, 0)
    accident = np.minimum(np.maximum(0.03 * speed + 0.4 * difficulty + np.random.normal(0, 0.3, (num,)), 0), 1)
    
    # Regress speed (Y) on difficulty (X) and accident (Z)
    indep = sm.add_constant(np.column_stack((difficulty, accident)))
    model = sm.OLS(speed, indep)
    results = model.fit()
    coefs_xz.append(results.params[1]) # Index 1 is difficulty (X)

np.mean(coefs_xz)

np.float64(-10.329336990107059)

Week 6

In [1]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# 1. Load the dataset
df = pd.read_csv('homework_6.1.csv')

# Separate treated (X=1) and untreated (X=0) items
treated = df[df['X'] == 1].reset_index(drop=True)
untreated = df[df['X'] == 0].reset_index(drop=True)

# 2. Fit NearestNeighbors models based on the confounder Z
nn_treated = NearestNeighbors(n_neighbors=1).fit(treated[['Z']])
nn_untreated = NearestNeighbors(n_neighbors=1).fit(untreated[['Z']])

# 3. Find counterfactuals
# For treated items: closest untreated item
_, dist_treated = nn_untreated.kneighbors(treated[['Z']])
te_treated = treated['Y'] - untreated.iloc[dist_treated.flatten()]['Y'].values

# For untreated items: closest treated item
_, dist_untreated = nn_treated.kneighbors(untreated[['Z']])
te_untreated = treated.iloc[dist_untreated.flatten()]['Y'].values - untreated['Y']

# 4. Calculate Treatment Effects
# Overall Average Treatment Effect (ATE)
total_items = len(te_treated) + len(te_untreated)
ate = (te_treated.sum() + te_untreated.sum()) / total_items

# Average Treatment Effect on the Treated (ATT)
att = te_treated.mean()

# Average Treatment Effect on the Untreated (ATU)
atu = te_untreated.mean()

# Optimal treatment effect (max TE across all untreated)
opt_te = te_untreated.max()

print(f"ATE: {ate}")
print(f"ATT: {att}")
print(f"ATU: {atu}")
print(f"Optimal TE: {opt_te}")

ATE: 1.6952701427497883
ATT: 1.8464085071912946
ATU: 1.5494765534751722
Optimal TE: 2.172469885577008
